In [1]:
import pandas as pd
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

In [2]:
lake = 'zug'

base_path = rf"/storage/alplakes_test/{lake}_100m_2025"
model = f'{lake}_2025'

input_folder = os.path.join(base_path, "outputs_swirl", "eddy_catalogues_final")

output_folder = os.path.join(base_path, "outputs_swirl", "ke_eddy")
os.makedirs(output_folder, exist_ok=True)

In [3]:
lake_csv_path = os.path.join(input_folder, "lake_characteristics.csv")

In [4]:
df_lake = pd.read_csv(lake_csv_path)
df_lake = df_lake.set_index('id', drop=False)
df_lake['date'] = pd.to_datetime(df_lake['date'])

# Depth csv

In [5]:
depths = df_lake[['depth_index', 'depth_[m]']].drop_duplicates().sort_values('depth_index').reset_index(drop=True)

In [6]:
thick = [depths['depth_[m]'].iloc[0] * 2]
for i in range(1, len(depths)):
    thick.append(depths['depth_[m]'].iloc[i] - depths['depth_[m]'].iloc[i-1])

depths['thickness_[m]'] = thick
depths.to_csv(os.path.join(base_path, "grid", "depths.csv"), index=False)

# Lake mask

In [3]:
sys.path.append('..//')
from utils_mitgcm import open_mitgcm_ds_from_config

In [4]:
mitgcm_config, ds = open_mitgcm_ds_from_config('..//config.json', model)

In [5]:
mask = ds['THETA'].isel(time=0).values > 0

In [6]:
np.save(os.path.join(base_path, "grid_metadata", "mask_lake.npy"), mask)

In [7]:
np.save(os.path.join(base_path, "grid", "mask_lake.npy"), mask)